# V-BOT Advanced v2 Training on Google Colab GPU

**5.85M-param Decoder-Only GPT** | 256-dim | 8 heads | 6 layers | 256-ctx | BPE 2048 | Dual-Domain SFT

### Step 1: Enable T4 GPU
1. **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**
2. Then **Run All** (`Ctrl+F9`)

> Expected: ~8-12 min on T4 GPU vs ~8 hrs on CPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('WARNING -- No GPU detected! Change runtime to T4 GPU.')


### Step 2: Upload Files

Upload from your local `projects/llm/` folder via the folder icon in the left sidebar:
- `combined_train.txt`, `qa_dataset.txt`, `bpe_tokenizer.json`
- `train_tokens.pt`, `val_tokens.pt` (optional, speeds up startup)


In [ ]:
import os
required = ['combined_train.txt', 'qa_dataset.txt', 'bpe_tokenizer.json']
missing  = [f for f in required if not os.path.exists(f)]
if missing:
    print('Missing:', missing, '-- opening upload dialog...')
    from google.colab import files
    files.upload()
else:
    print('All files found!')
for fn in required:
    if os.path.exists(fn):
        print(f'  {fn:30s} {os.path.getsize(fn):>12,} bytes')


### Step 3: BPE Tokenizer + Hyperparameters + Data

In [ ]:
import json, math, os, time
import torch
import torch.nn as nn
from torch.nn import functional as F
from collections import defaultdict

class BPETokenizer:
    def __init__(self, target_vocab_size=2048):
        self.target_vocab_size = target_vocab_size
        self.merges = {}
        self.vocab  = {}
        self.vocab_size = 0

    def train(self, text):
        tokens = list(text.encode("utf-8"))
        vocab  = {i: bytes([i]) for i in range(256)}
        vocab[256] = b"<|endoftext|>"
        next_id = 257
        for step in range(self.target_vocab_size - 257):
            counts = defaultdict(int)
            for a, b in zip(tokens, tokens[1:]):
                counts[(a, b)] += 1
            if not counts:
                break
            best = max(counts, key=counts.__getitem__)
            if counts[best] < 2:
                break
            out, i = [], 0
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    out.append(next_id); i += 2
                else:
                    out.append(tokens[i]); i += 1
            tokens = out
            vocab[next_id] = vocab[best[0]] + vocab[best[1]]
            self.merges[best] = next_id
            next_id += 1
            if step % 300 == 0:
                print(f"  Merge {step}/{self.target_vocab_size-257}")
        self.vocab = vocab
        self.vocab_size = next_id

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        while True:
            pairs = set(zip(tokens, tokens[1:]))
            cands = {p: self.merges[p] for p in pairs if p in self.merges}
            if not cands:
                break
            best   = min(cands, key=lambda p: cands[p])
            new_id = cands[best]
            out, i = [], 0
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    out.append(new_id); i += 2
                else:
                    out.append(tokens[i]); i += 1
            tokens = out
        return tokens

    def decode(self, ids):
        return b"".join(self.vocab.get(i, b"") for i in ids).decode("utf-8", errors="replace")

    def save(self, path):
        with open(path, "w") as f:
            json.dump({
                "vocab_size": self.vocab_size,
                "merges": {f"{a},{b}": c for (a,b),c in self.merges.items()},
                "vocab":  {str(k): list(v) for k,v in self.vocab.items()},
            }, f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        tok = cls()
        tok.vocab_size = data["vocab_size"]
        tok.merges = {tuple(map(int, k.split(","))): v for k,v in data["merges"].items()}
        tok.vocab  = {int(k): bytes(v) for k,v in data["vocab"].items()}
        return tok


TOKENIZER_PATH = "bpe_tokenizer.json"
if os.path.exists(TOKENIZER_PATH):
    tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    print(f"Loaded BPE tokenizer | vocab_size: {tokenizer.vocab_size}")
else:
    with open("combined_train.txt", encoding="utf-8") as f:
        raw = f.read().replace("\r\n", "\n")
    tokenizer = BPETokenizer(2048)
    tokenizer.train(raw)
    tokenizer.save(TOKENIZER_PATH)
    print(f"Trained tokenizer | vocab_size: {tokenizer.vocab_size}")

vocab_size = tokenizer.vocab_size
encode     = tokenizer.encode
decode     = tokenizer.decode

torch.manual_seed(1337)
device        = "cuda" if torch.cuda.is_available() else "cpu"
n_embd        = 256
n_head        = 8
n_layer       = 6
block_size    = 256
dropout       = 0.1
batch_size    = 64
max_iters     = 3000
eval_interval = 200
eval_iters    = 50
learning_rate = 1.0e-3
min_lr        = 1.0e-4
warmup_iters  = 150
weight_decay  = 0.1
max_norm      = 1.0
print(f"Device: {device.upper()} | {n_layer}L {n_head}H {n_embd}D {block_size}ctx | {max_iters} iters")

if os.path.exists("train_tokens.pt") and os.path.exists("val_tokens.pt"):
    train_data = torch.load("train_tokens.pt", map_location="cpu")
    val_data   = torch.load("val_tokens.pt",   map_location="cpu")
    print(f"Loaded tensors: train={len(train_data):,} | val={len(val_data):,} tokens")
else:
    with open("combined_train.txt", encoding="utf-8") as f:
        raw_text = f.read().replace("\r\n", "\n")
    print("Tokenizing (may take ~30s)...")
    data_tensor = torch.tensor(encode(raw_text), dtype=torch.long)
    split_idx   = int(0.9 * len(data_tensor))
    train_data  = data_tensor[:split_idx]
    val_data    = data_tensor[split_idx:]
    print(f"Tokenized: train={len(train_data):,} | val={len(val_data):,}")

qa_samples = []
if os.path.exists("qa_dataset.txt"):
    with open("qa_dataset.txt", encoding="utf-8") as f:
        qa_raw = f.read().replace("\r\n", "\n")
    for pair in [p.strip() for p in qa_raw.split("\n\n") if "Assistant:" in p]:
        up, ap = pair.split("Assistant:", 1)
        pids   = encode(up + "Assistant:")
        aids   = encode(" " + ap.strip() + " <|endoftext|>")
        full   = (pids + aids)[:block_size]
        x, y   = full[:-1], full[1:]
        plen   = len(pids) - 1
        ym     = [-100] * plen + y[plen:]
        pad    = block_size - len(x)
        if pad > 0:
            x  += [256] * pad
            ym += [-100] * pad
        qa_samples.append((
            torch.tensor(x,  dtype=torch.long),
            torch.tensor(ym, dtype=torch.long),
        ))
    print(f"Loaded {len(qa_samples)} QA SFT pairs")
else:
    print("No qa_dataset.txt -- corpus-only training")


### Step 4: Model + Training (Decoupled AdamW + Cosine LR + SFT)

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = (q @ k.transpose(-2,-1)) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T,:T]==0, float("-inf"))
        return self.dropout(F.softmax(wei, dim=-1)) @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd), nn.GELU(),
            nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        hs = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, hs)
        self.ffwd = FeedForward()
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[TransformerBlock() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = (self.token_embedding_table(idx) +
             self.position_embedding_table(torch.arange(T, device=idx.device)))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T,C), targets.view(B*T), ignore_index=-100)
        return logits, loss
    def generate(self, idx, max_new_tokens, temperature=0.6, top_k=20):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:,-1,:] / max(temperature, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:,[-1]]] = -float("Inf")
            idx_next = torch.multinomial(F.softmax(logits, dim=-1), 1)
            if idx_next.item() == 256:
                break
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"V-BOT Advanced v2 on {device.upper()} | {total:,} params ({total/1e6:.2f}M)")

def get_batch(split):
    half = batch_size // 2
    data = train_data if split == "train" else val_data
    if qa_samples and split == "train":
        qi   = torch.randint(len(qa_samples), (half,))
        x_qa = torch.stack([qa_samples[i][0] for i in qi])
        y_qa = torch.stack([qa_samples[i][1] for i in qi])
        si   = torch.randint(len(data)-block_size-1, (half,))
        x_c  = torch.stack([data[i:i+block_size]     for i in si])
        y_c  = torch.stack([data[i+1:i+block_size+1] for i in si])
        x, y = torch.cat([x_qa,x_c]), torch.cat([y_qa,y_c])
    else:
        ix   = torch.randint(len(data)-block_size-1, (batch_size,))
        x    = torch.stack([data[i:i+block_size]     for i in ix])
        y    = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it+1) / (warmup_iters+1)
    if it > max_iters:
        return min_lr
    ratio = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses, accs = [], []
        for _ in range(eval_iters):
            xb, yb       = get_batch(split)
            logits, loss = model(xb, yb)
            losses.append(loss.item())
            B, T, V = logits.shape
            lf, tf  = logits.view(B*T,V), yb.view(B*T)
            mask = tf != -100
            if mask.any():
                accs.append((lf[mask].argmax(-1) == tf[mask]).float().mean().item())
        out[split] = {
            "loss": sum(losses)/len(losses),
            "acc":  sum(accs)/len(accs) if accs else 0.0,
        }
    model.train()
    return out

pdict  = {n: p for n,p in model.named_parameters() if p.requires_grad}
decay  = [p for n,p in pdict.items() if p.dim() >= 2]
nodec  = [p for n,p in pdict.items() if p.dim() <  2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": weight_decay},
     {"params": nodec, "weight_decay": 0.0}],
    lr=learning_rate, betas=(0.9, 0.95), eps=1e-8)
print(f"AdamW | decay={sum(p.numel() for p in decay):,} | nodecay={sum(p.numel() for p in nodec):,}")

print(f"\nStarting {max_iters} iters | eval every {eval_interval} steps")
print("="*80)
t0 = time.time()
best_val_loss = float("inf")

for it in range(max_iters):
    lr = get_lr(it)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    if it % eval_interval == 0 or it == max_iters - 1:
        m   = estimate_loss()
        tl  = m["train"]["loss"]
        vl  = m["val"]["loss"]
        acc = m["val"]["acc"] * 100
        print(f"Iter {it:4d}/{max_iters} | "
              f"Train {tl:.4f} PPL={math.exp(min(tl,20)):.1f} | "
              f"Val {vl:.4f} PPL={math.exp(min(vl,20)):.1f} | "
              f"Acc {acc:.1f}% | LR {lr:.2e} | {time.time()-t0:.0f}s")
        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), "model_best.pt")

    xb, yb       = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
    optimizer.step()

print("="*80)
print(f"Done in {(time.time()-t0)/60:.1f} min | best val loss: {best_val_loss:.4f}")
torch.save(model.state_dict(), "model.pt")
print("Saved: model.pt  |  model_best.pt")


### Step 5: Generation Test

In [ ]:
model.eval()
test_prompts = [
    "User: What is your name?\nAssistant:",
    "User: What is Vishwas AI?\nAssistant:",
    "User: How does a transformer model generate text?\nAssistant:",
    "User: What is PyTorch?\nAssistant:",
    "User: Who is William Shakespeare?\nAssistant:",
]
print("="*70 + "\nV-BOT ADVANCED v2 -- GENERATION TEST\n" + "="*70)
with torch.no_grad():
    for p in test_prompts:
        ctx   = torch.tensor([encode(p)], dtype=torch.long, device=device)
        out   = model.generate(ctx, max_new_tokens=120, temperature=0.5, top_k=15)[0].tolist()
        reply = decode(out)[len(p):]
        for stop in ["\nUser:", "<|endoftext|>"]:
            if stop in reply:
                reply = reply.split(stop)[0]
        print(f"\n>>> {p.strip()}\n{reply.strip()}")
print("\n" + "="*70)


### Step 6: Download Checkpoint

Download and replace your local `projects/llm/model.pt`.


In [ ]:
from google.colab import files
if os.path.exists("model_best.pt"):
    print("Downloading model_best.pt (best val loss)...")
    files.download("model_best.pt")
print("Downloading model.pt (final checkpoint)...")
files.download("model.pt")
print("Done! Replace your local projects/llm/model.pt with the downloaded file.")
